# 00. 하네스와 현재 그래프 감사

## Question
어떻게 기존 서비스와 실험을 보존하면서 다음 세션에서도 이유와 결과를 이해하고 이어갈 수 있을까요?

## Hypothesis
기억 파일과 작은 노트북 추출 도구면 충분합니다. 별도의 실험 플랫폼은 아직 필요하지 않습니다.

## Baseline
Before 코드: `08d686b31904b90e9aa5d84e8803e03f51567cc2`.
서비스와 하네스 v1은 변경하지 않습니다. 모델·법령 API·Qdrant 호출을 하지 않습니다.
기존 14항목은 독립 holdout이 아니며 기존 recall은 정답 하나라도 찾았는지 보는 Hit입니다.

## Experiment
설명 → 구현 → 작은 예 → 테스트 순서로 읽습니다. 추출 도구는 표준 라이브러리만 사용합니다.
`export`는 옮길 코드 셀 표시, `export_to`는 저장소 안의 목적지입니다.
[Jupyter 공식 형식](https://nbformat.readthedocs.io/en/latest/format_description.html)의 셀 metadata를 사용합니다.
새 프레임워크나 노트북 import 도구를 설치하지 않습니다.


In [1]:
from pathlib import Path

# 저장소 루트 또는 이 노트북 폴더에서 실행할 수 있습니다.
ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "pyproject.toml").is_file() and (path / "harness").is_dir())
NOTEBOOK = ROOT / "notebooks/redesign/00_harness_and_audit.ipynb"
print("로컬 저장소와 감사 노트북을 찾았습니다.")


로컬 저장소와 감사 노트북을 찾았습니다.


### 구현: 코드를 실행하지 않고 추출하기
선택한 셀만 모아 Python 문법을 확인합니다. 경로가 저장소 밖으로 나가거나 수동 작성 파일을 덮어쓰면 중단합니다.
먼저 모든 대상을 확인한 다음 씁니다. `--check`는 파일을 바꾸지 않습니다.
같은 파일의 셀 순서는 노트북 순서이며 두 노트북이 같은 파일을 소유할 수 없습니다.


In [2]:
"""노트북 export 셀을 실행 없이 추출하고 Python 파일과 비교합니다."""

import argparse
import ast
import json
from pathlib import Path


def render_exports(notebook: Path, root: Path) -> dict[Path, str]:
    """노트북에서 {안전한 목적지: Python 소스}를 반환합니다. 실행/쓰기는 하지 않습니다."""
    root = root.resolve()
    notebook = notebook.resolve()
    relative = notebook.relative_to(root).as_posix()
    data = json.loads(notebook.read_text(encoding="utf-8-sig"))
    groups = {}
    for cell in data["cells"]:
        metadata = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "export" not in metadata.get("tags", []):
            continue
        name = metadata.get("export_to")
        if not isinstance(name, str) or not name:
            raise ValueError("export 셀에는 export_to 경로가 필요합니다.")
        path = Path(name)
        # resolve는 상위 경로와 심볼릭 링크를 포함한 실제 목적지를 확인합니다.
        target = (root / path).resolve()
        if path.is_absolute() or not target.is_relative_to(root):
            raise ValueError(f"저장소 밖으로 추출할 수 없습니다: {name}")
        inside = target.relative_to(root)
        if inside.parts[0] not in {"app", "harness", "tests"} or target.suffix != ".py":
            raise ValueError(f"app/harness/tests의 Python 파일만 허용합니다: {name}")
        source = cell.get("source", [])
        text = source if isinstance(source, str) else "".join(source)
        groups.setdefault(target, []).append(text.rstrip())
    if not groups:
        raise ValueError("export 코드 셀이 없습니다.")
    result = {}
    for target, parts in groups.items():
        # 첫 줄의 소유권 표시로 다른 원본의 파일 덮어쓰기를 막습니다.
        header = f"# Generated from {relative}; edit the notebook, not this file.\n"
        source = header + "\n\n".join(parts) + "\n"
        ast.parse(source, filename=str(target))
        result[target] = source
    return result


def sync_exports(notebooks: list[Path], root: Path, *, write: bool = False) -> list[str]:
    """차이 목록을 반환합니다. write=True이면 모든 검증 후 추출본을 갱신합니다."""
    root = root.resolve()
    rendered = {}
    for notebook in notebooks:
        for target, source in render_exports(notebook, root).items():
            if target in rendered:
                raise ValueError(f"여러 노트북이 같은 파일을 지정했습니다: {target}")
            rendered[target] = source
    changed = []
    for target, source in rendered.items():
        existing = target.read_text(encoding="utf-8") if target.exists() else None
        owner = source.splitlines()[0]
        if existing is not None and (not existing.splitlines() or existing.splitlines()[0] != owner):
            raise ValueError(f"수동 작성 파일 또는 다른 노트북 소유 파일입니다: {target}")
        if existing != source:
            changed.append(target.relative_to(root).as_posix())
    if write:
        for target, source in rendered.items():
            if target.relative_to(root).as_posix() in changed:
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_text(source, encoding="utf-8", newline="\n")
    return changed


def main() -> int:
    """--check는 차이가 있으면 종료 코드 1, --write는 추출본을 갱신합니다."""
    parser = argparse.ArgumentParser(description=__doc__)
    mode = parser.add_mutually_exclusive_group(required=True)
    mode.add_argument("--check", action="store_true")
    mode.add_argument("--write", action="store_true")
    parser.add_argument("notebooks", nargs="*", help="저장소 기준 노트북 경로")
    args = parser.parse_args()
    root = Path(__file__).resolve().parents[1]
    names = args.notebooks or ["notebooks/redesign/00_harness_and_audit.ipynb"]
    changed = sync_exports([root / name for name in names], root, write=args.write)
    print("Changed: " + ", ".join(changed) if changed else "Notebook exports match.")
    return int(bool(changed) and not args.write)


if __name__ == "__main__" and "__file__" in globals():
    raise SystemExit(main())


### 테스트: 위험한 실수를 막는가?
임시 폴더에서 실행하며 기존 서비스와 외부 시스템에 접근하지 않습니다.
코드 셀이 실행되지 않는지, 경로 탈출·수동 파일 덮어쓰기가 차단되는지, 차이를 감지하는지 확인합니다.
아래 추출 셀은 함수 정의 셀 뒤에 실행합니다. CLI 코드는 노트북에서는 실행되지 않습니다.


In [3]:
# 위에서 정의한 함수로 추출합니다. export 셀을 실행하는 호출은 아닙니다.
changed = sync_exports([NOTEBOOK], ROOT, write=True)
print("추출 결과:", changed or "변경 없음")
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


추출 결과: 변경 없음


In [4]:
"""노트북 추출의 경로·소유권·재현성을 임시 파일로 검증합니다."""

import json
import tempfile
import unittest
from pathlib import Path

from harness.notebook_tools import render_exports, sync_exports


class NotebookExportTests(unittest.TestCase):
    """파일 추출에서 보존해야 할 경계를 검증합니다."""

    def setUp(self):
        """각 테스트에 독립적인 임시 폴더를 만듭니다."""
        self.temp = tempfile.TemporaryDirectory()
        self.addCleanup(self.temp.cleanup)
        self.root = Path(self.temp.name)
        self.nb = self.root / "example.ipynb"

    def notebook(self, target="harness/example.py", source="value = 1", extra=None):
        """지정한 목적지·코드로 작은 노트북을 만들어 경로를 반환합니다."""
        cells = [{"cell_type": "code", "metadata": {"tags": ["export"], "export_to": target},
                  "source": source}]
        cells.extend(extra or [])
        self.nb.write_text(json.dumps({"cells": cells}), encoding="utf-8")
        return self.nb

    def test_roundtrip_and_drift(self):
        """첫 쓰기 후 일치하며 수동 변경은 check에서 감지해야 합니다."""
        nb = self.notebook()
        self.assertEqual(sync_exports([nb], self.root), ["harness/example.py"])
        self.assertFalse((self.root / "harness").exists())
        sync_exports([nb], self.root, write=True)
        self.assertEqual(sync_exports([nb], self.root), [])
        target = self.root / "harness/example.py"
        target.write_text(target.read_text() + "value = 2\n")
        self.assertEqual(sync_exports([nb], self.root), ["harness/example.py"])

    def test_does_not_execute_and_skips_untagged(self):
        """예외를 던지는 셀도 문법만 검사하며 미지정 셀은 추출하지 않습니다."""
        nb = self.notebook(source="raise RuntimeError('must not execute')", extra=[
            {"cell_type": "code", "metadata": {}, "source": "not valid Python !"}])
        result = render_exports(nb, self.root)
        self.assertIn("raise RuntimeError", next(iter(result.values())))

    def test_blocks_unsafe_paths(self):
        """상위 경로·절대 경로·허용하지 않은 목적지를 거절합니다."""
        for target in ["../outside.py", str(self.root / "absolute.py"), ".env", "docs/x.py"]:
            with self.subTest(target=target), self.assertRaises(ValueError):
                render_exports(self.notebook(target), self.root)

    def test_preserves_manual_file(self):
        """소유권 표시 없는 기존 파일을 덮어쓰지 않습니다."""
        target = self.root / "harness/example.py"
        target.parent.mkdir()
        target.write_text("original = True")
        with self.assertRaises(ValueError):
            sync_exports([self.notebook()], self.root, write=True)
        self.assertEqual(target.read_text(), "original = True")

    def test_validates_all_before_writing(self):
        """뒤쪽 문법 오류가 앞 파일의 부분 갱신을 만들지 않습니다."""
        nb = self.notebook(extra=[{"cell_type": "code", "metadata": {
            "tags": ["export"], "export_to": "harness/broken.py"}, "source": "def broken("}])
        with self.assertRaises(SyntaxError):
            sync_exports([nb], self.root, write=True)
        self.assertFalse((self.root / "harness").exists())

    def test_duplicate_owners_and_missing_target(self):
        """중복 소유권·목적지 누락을 조용히 무시하지 않습니다."""
        nb = self.notebook()
        with self.assertRaises(ValueError):
            sync_exports([nb, nb], self.root, write=True)
        self.assertFalse((self.root / "harness").exists())
        with self.assertRaises(ValueError):
            render_exports(self.notebook(target=""), self.root)

    def test_multiple_cells_keep_order(self):
        """같은 목적지의 셀은 노트북 순서로 결합합니다."""
        nb = self.notebook(source="first = 1", extra=[{"cell_type": "code", "metadata": {
            "tags": ["export"], "export_to": "harness/example.py"}, "source": ["second = first + 1"]}])
        result = next(iter(render_exports(nb, self.root).values()))
        self.assertLess(result.index("first = 1"), result.index("second ="))


In [5]:
suite = unittest.defaultTestLoader.loadTestsFromTestCase(NotebookExportTests)
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful(), "추출 도구 검증 실패"
assert sync_exports([NOTEBOOK], ROOT) == [], "노트북과 Python이 다릅니다."


test_blocks_unsafe_paths (__main__.NotebookExportTests.test_blocks_unsafe_paths)
상위 경로·절대 경로·허용하지 않은 목적지를 거절합니다. ... 

ok


test_does_not_execute_and_skips_untagged (__main__.NotebookExportTests.test_does_not_execute_and_skips_untagged)
예외를 던지는 셀도 문법만 검사하며 미지정 셀은 추출하지 않습니다. ... 

ok


test_duplicate_owners_and_missing_target (__main__.NotebookExportTests.test_duplicate_owners_and_missing_target)
중복 소유권·목적지 누락을 조용히 무시하지 않습니다. ... 

...

test_roundtrip_and_drift (__main__.NotebookExportTests.test_roundtrip_and_drift)
첫 쓰기 후 일치하며 수동 변경은 check에서 감지해야 합니다. ... 

ok


test_validates_all_before_writing (__main__.NotebookExportTests.test_validates_all_before_writing)
뒤쪽 문법 오류가 앞 파일의 부분 갱신을 만들지 않습니다. ... 

ok


----------------------------------------------------------------------
Ran 7 tests in 0.104s

OK


## 현재 그래프 감사 — 정적 확인, 동작 재현은 단계 3

```mermaid
flowchart LR
  Q[채팅 요청] --> P[개인정보 검사]
  P --> U[의도·재작성·확답 차단]
  U --> R[dense 검색·문서 유형 필터]
  R --> C[최상위 점수 검사]
  C --> I[인용 준비]
  I --> G[sLLM 생성]
  G --> S[SSE 응답]
```

| 관찰 | 근거 위치 | 쉬운 설명·다음 실험 |
|---|---|---|
| 이력·계약서 컨텍스트 전달 누락 | app/services/chat_service.py: run_preformat(query) | 받은 정보를 다음 단계에 넘기지 않음; 전달·격리 검증 |
| 생성 답변이 messages에 추가되지 않음 | app/graph/nodes/generator.py | 후속 대화에 답변을 기억하는 주체 확정 |
| 비스트리밍 생성 후 인용, 스트리밍 생성 전 인용 | app/graph/pipeline.py | 같은 근거·출력 계약인지 확인 |
| 유형 필터의 근거 누락 가능성 | retrieve.py, understand_query.py | 필터 유무 비교, 아직 성능 결함으로 확정하지 않음 |
| top-1로 답변 가능 여부 판단 | check_relevance.py | 하나가 비슷해도 모든 조건의 근거가 충분한 것은 아님 |
| 예전 formatter 등 잔존 | pipeline.py의 노드 목록과 import | 파일 존재와 실행 구분, helper 사용 확인 후 정리 |
| 문자 수를 토큰 수로 집계 | chat_service.py | 글자 수는 모델 토큰 수가 아님; 실제 usage 기록 필요 |
| 느린 소비자 queue 처리 | chat_service.py: put_nowait | 버퍼가 찰 때 종료 이벤트까지 전달되는지 재현 |

이 관찰은 LangGraph 라이브러리 자체의 버그를 증명하지 않습니다. 서비스는 수정하지 않았습니다.

## Metrics
이번 단계는 추출 도구의 7개 로컬 테스트와 노트북/추출본 일치를 검증합니다.
RAG 품질 지표는 [평가 규약](evaluation_protocol.md)에 정의했으며 아직 측정하지 않았습니다.
[RAGAS 원 논문](https://arxiv.org/abs/2309.15217)은 검색·생성 평가 분리의 근거이며 judge를 설치하지 않습니다.

## Result
위 셀의 출력이 실제 검증 결과입니다. 모델 성능·비용·법률 정확도 실측은 없습니다.

## Failure Cases
경로 탈출·수동 파일 덮어쓰기·잘못된 문법·중복 소유권·추출본 불일치를 작은 예로 검증합니다.
서비스 상태 전달·인용·SSE 실패 재현은 아직 실행하지 않았습니다.

## Decision
기억 파일과 이 작은 추출 도구를 사용합니다. 기존 v1·서비스·평가셋은 보존합니다.

## Critical Review
- 필요성: 노트북과 서비스 코드가 달라지는 실수를 감지합니다.
- 단순성: 표준 라이브러리와 함수 3개로 구성하며 새 프레임워크를 추가하지 않습니다.
- 한계: 쓰기 도중 OS 오류의 트랜잭션 복구나 완전한 비밀 탐지는 제공하지 않습니다. Git diff를 검토합니다.
- 다음: Git 반영 승인과 별도 단계 1 승인을 받은 후 평가셋·지표 함수를 구현합니다.


## 로컬 검증 기록 — 2026-10-16
별도 config/metrics/review 파일의 중복 내용을 이곳에 합쳤습니다. 모델 성능 실험이 아닙니다.

- 기준 코드: 08d686b31904b90e9aa5d84e8803e03f51567cc2. 구현 커밋: 7ecee7b.
- 새 커널 전체 실행·nbformat·export 일치 통과.
- pytest tests/unit/test_notebook_tools.py tests/unit/test_sse.py -q -p no:cacheprovider:
  **11 passed, 4 subtests passed (52.05s)**.
- 저장소 경계·파일 소유권·쓰기 전 문법 검증을 확인했습니다. OS 쓰기 실패의 다중 파일 원자성은 제공하지 않습니다.
- Windows stdin 인코딩으로 한글이 손상된 문제는 UTF-8로 복구한 뒤 재실행했습니다.
- 당시 모델 호출·패키지 설치·서비스 변경은 없었습니다. API 전체 통합·GPU·원격 DB·SSE 부하/취소는 미검증입니다.
- 인증키 점검은 제한된 패턴 검사이며 전체 보안 감사를 뜻하지 않습니다.
- PR #5: https://github.com/bigmooon/aigo-ai/pull/5 (병합 확인).
- 당시 파일 전체는 Git 이력에서 확인합니다. 삭제한 해시 목록만으로 미커밋 파일을 복원할 수 있다고 주장하지 않습니다.
